In [1]:
import BECancerResistome
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re
import math
from sklearn.preprocessing import OrdinalEncoder

In [2]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Load Data

In [ ]:
zscores_plasmid_unambiguous_VEPannotated = pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-processed-plasmid.csv")
zscores_plasmid_unambiguous_VEPannotated_all_features =pd.read_csv("data/6_VEP/zscores-unambiguous-VEPannotated-plasmid.csv")

In [ ]:
zscores_plasmid_unambiguous_VEPannotated.head()

In [ ]:
zscores_plasmid_unambiguous_VEPannotated_all_features.head()

### Filter for Plasmid vs Control

In [10]:
#filter for 'Drug' == 'Control' or 'DO'
zscores_plasmid_vs_control_annotated = zscores_plasmid_unambiguous_VEPannotated[
    (zscores_plasmid_unambiguous_VEPannotated['Drug'] == 'Control') |
    (zscores_plasmid_unambiguous_VEPannotated['Drug'] == 'DO')
]

In [ ]:
zscores_plasmid_vs_control_annotated.head()

In [ ]:
zscores_plasmid_vs_control_annotated['Cell_Line'].value_counts()

In [ ]:
zscores_plasmid_vs_control_annotated['Hit_class'].value_counts()

#### Filter for hits to plot (with all features dataset)

In [33]:
#Filter zscores_plasmid_unambiguous_VEPannotated_all_features for 'Hit_class' == 'negative' or 'positive' and 'Drug' == 'Control' or 'DO'
zscores_plasmid_vs_control_annotated_all_features_hits = zscores_plasmid_unambiguous_VEPannotated_all_features[
    ((zscores_plasmid_unambiguous_VEPannotated_all_features['Hit_class'] == 'negative') |
    (zscores_plasmid_unambiguous_VEPannotated_all_features['Hit_class'] == 'positive')) &
    ((zscores_plasmid_unambiguous_VEPannotated_all_features['Drug'] == 'Control') |
    (zscores_plasmid_unambiguous_VEPannotated_all_features['Drug'] == 'DO'))
]

In [ ]:
zscores_plasmid_vs_control_annotated_all_features_hits.head()

In [ ]:
zscores_plasmid_vs_control_annotated_all_features_hits['Mutation Category'].value_counts()

# Check Correlation 

In [ ]:
meta_cols=[
    'Target Transcript ID', 'RefSeq match transcript (MANE Select)', 'Guide', 'Editor', 'Gene', 'Cell_Line','Drug', 'Amino Acid Edits', 
    'Source', 'Hit_class'
]

training_features_df = zscores_plasmid_vs_control_annotated.drop(columns=meta_cols)

print(f"Shape of feature dataset: {training_features_df.shape}")

In [ ]:
# Compute correlation matrix
corr_matrix = training_features_df.corr()
corr_matrix

In [ ]:
# Plot heatmap
plt.figure(figsize=(9,7))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, cbar_kws={'label': 'Person Correlation Coefficient, r'})
plt.show()

In [ ]:
# Top 5 features most positively correlated with z-score
top_5_positive_corr = corr_matrix['zscore'].sort_values(ascending=False).head(6)
print(f'Top 5 features most positively correlated with z-score:')
print(top_5_positive_corr)  # Including 'zscore' itself


In [ ]:
# Top 5 features most negatively correlated with z-score
top_5_negative_corr = corr_matrix['zscore'].sort_values(ascending=True).head(5)
print(f'Top 5 features most negatively correlated with z-score:')
print(top_5_negative_corr)

In [21]:
def plot_gene_conditions(
    df: pd.DataFrame,
    gene: str,
    vep_col: str,
    colorbar_label: str = None,
    condition_cols=("Cell_Line", "Drug"),
    point_size: int = 18,
    cmap: str = "viridis",
    max_cols: int = 3,
):
    # Filter by gene
    data = df[df["Gene"] == gene].copy()
    if data.empty:
        print(f"No data for gene {gene}")
        return

    # Build a readable condition string
    plus_sep = " + "
    base = data.loc[:, condition_cols].astype(str).agg(plus_sep.join, axis=1)
    if "Source" in data.columns:
        src = data["Source"].fillna("")
        data["Condition"] = np.where(src.eq(""), base, base + " | " + src.astype(str))
    else:
        data["Condition"] = base

    conditions = data["Condition"].unique()
    n = len(conditions)

    # Grid geometry + figure size that scales with n
    ncols = min(max_cols, n)
    nrows = math.ceil(n / ncols)
    fig_w = 4.5 * ncols
    fig_h = 3.8 * nrows

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(fig_w, fig_h),
        constrained_layout=False,
        squeeze=False
    )
    fig.subplots_adjust(hspace=0.6, wspace=0.25)

    # Color scaling shared across all subplots
    vmin = data[vep_col].min()
    vmax = data[vep_col].max()

    # Global symmetric y-axis across all subplots
    y_absmax = max(abs(data["zscore"].min()), abs(data["zscore"].max()))
    y_min, y_max = -y_absmax, y_absmax

    # Plot each condition
    last_sc = None
    for ax, cond in zip(axes.ravel(), conditions):
        sub = data[data["Condition"] == cond]
        last_sc = ax.scatter(
            sub["Protein_position"],
            sub["zscore"],
            c=sub[vep_col],
            s=point_size,
            cmap=cmap,
            vmin=vmin, vmax=vmax,
            alpha=0.8,
            edgecolors="none"
        )
        ax.set_title(cond, fontsize=10)
        ax.set_xlabel("Protein position", fontsize=9)
        ax.set_ylabel("Z-score", fontsize=9)
        ax.tick_params(labelsize=8)

        # Force symmetric y-limits centered at 0
        ax.set_ylim(y_min, y_max)
        ax.axhline(0, color="gray", lw=0.8, linestyle="--")  # optional zero reference line

    # Hide unused axes
    for ax in axes.ravel()[n:]:
        ax.set_visible(False)

    # Shared colorbar
    if last_sc is not None:
        cbar = fig.colorbar(
            last_sc,
            ax=axes.ravel().tolist(),
            location="right",
            shrink=0.5,
            pad=0.02,
            aspect=25
        )
        cbar.set_label(colorbar_label if colorbar_label else vep_col, fontsize=10)
        cbar.ax.tick_params(labelsize=8)

    plt.show()

In [ ]:
plot_gene_conditions(
    df=zscores_plasmid_vs_control_annotated_all_features_hits,
    gene="PIK3CA",
    vep_col="AlphaMissense_score",
    colorbar_label="AlphaMissense score",
    condition_cols=["Cell_Line", "Drug"],
)